# Guardian Football News — Sentence Embeddings

Download the latest football articles from The Guardian, embed each sentence, and find sentences most similar to a query.

In [17]:
## Cell 1 — Download latest football articles from The Guardian

import os
import pickle
import requests
from dotenv import load_dotenv

load_dotenv()
GUARDIAN_API_KEY = os.getenv("GUARDIAN_API_KEY")

def fetch_guardian_football(api_key, page_size=50):
    url = "https://content.guardianapis.com/football"
    params = {
        "api-key": api_key,
        "page-size": page_size,
        "order-by": "newest",
        "show-fields": "headline,bodyText",
    }
    response = requests.get(url, params=params)
    response.raise_for_status()
    results = response.json()["response"]["results"]

    articles = []
    for item in results:
        fields = item.get("fields", {})
        headline = fields.get("headline", "(no headline)")
        body = fields.get("bodyText", "")
        if body:
            articles.append({"headline": headline, "body": body})
    return articles

articles = fetch_guardian_football(GUARDIAN_API_KEY)
print(f"Downloaded {len(articles)} articles.")
for a in articles[:3]:
    print(f"  • {a['headline']}")

# Save articles (with headlines) for later use
with open("guardian_football_articles.pkl", "wb") as f:
    pickle.dump(articles, f)


Downloaded 50 articles.
  • Nico O’Reilly’s fearless quality exposes collapsing Arsenal’s title credentials
  • Guardiola calls for focus in title run-in after Haaland cuts down Arsenal
  • Manchester City 2-1 Arsenal: Premier League – as it happened


In [ ]:
## Cell 2 — Split articles into sentences

import re

def split_into_sentences(text):
    # Split on sentence-ending punctuation followed by whitespace and a capital letter
    sentences = re.split(r'(?<=[.!?])\s+(?=[A-Z])', text.strip())
    return [s.strip() for s in sentences if len(s.strip()) > 20]

all_sentences = []
for article in articles:
    body = article["body"] if isinstance(article, dict) else article
    all_sentences.extend(split_into_sentences(body))

print(f"Total sentences: {len(all_sentences)}")
print("Sample sentences:")
for s in all_sentences[:5]:
    print(" •", s)


Total sentences: 3101
Sample sentences:
 • It’s not over, not over, not over yet.
 • Although, let’s be honest, it kind of is over.
 • Isn’t it, don’t you think, at the end of a day when Manchester City and Arsenal dished up the one thing nobody was expecting at the Etihad Stadium, a thrillingly open game of attacking football?
 • There were three images at the final whistle that seemed to capture the essence of City’s 2-1 win here, and not just in terms of the game, but the balance of energy, feeling, vibes.
 • The first was the sight of Erling Haaland marching around on a rabble-rousing victory lap, golden tresses dangling free, evening sun glowing across his slabbed and rippling chest, like a beautiful mermaid goddess, but a mermaid goddess who only eats protein and raw milk and does 600 sit‑ups a day.


In [ ]:
## Cell 3 — Embed the sentences with Azure OpenAI

from openai import AzureOpenAI
from dotenv import find_dotenv
from urllib.parse import urlparse
import numpy as np

# Parse .env manually — some keys have leading spaces which confuse dotenv
def read_env(path):
    env = {}
    with open(path) as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#") or "=" not in line:
                continue
            key, _, val = line.partition("=")
            env[key.strip()] = val.strip().strip('"').strip("'")
    return env

env = read_env(find_dotenv())

# Extract base URL (strip /openai/v1 path if present)
raw_base = env.get("GPT_BASE", "")
parsed = urlparse(raw_base)
azure_endpoint = f"{parsed.scheme}://{parsed.netloc}"

api_key     = env.get("GPT_KEY", "")
api_version = env.get("GPT_VERSION", "2025-04-01-preview")
EMBED_MODEL = env.get("GPT_EMBEDDINGS_MODEL", "text-embedding-3-large-context-course-2026-afthibara")

client = AzureOpenAI(
    azure_endpoint=azure_endpoint,
    api_key=api_key,
    api_version=api_version,
)

BATCH_SIZE = 100

def embed_texts(texts, model=EMBED_MODEL, batch_size=BATCH_SIZE):
    all_embeddings = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i : i + batch_size]
        response = client.embeddings.create(input=batch, model=model)
        batch_embeddings = [item.embedding for item in response.data]
        all_embeddings.extend(batch_embeddings)
    return np.array(all_embeddings, dtype="float32")

print(f"Using endpoint : {azure_endpoint}")
print(f"Using model    : {EMBED_MODEL}")
print(f"Embedding {len(all_sentences)} sentences …")
embeddings = embed_texts(all_sentences)
print(f"Embedding matrix shape: {embeddings.shape}")

# Save the sentence and their embeddings together for later use
import pickle
with open("sentences_and_embeddings.pkl", "wb") as f:
    pickle.dump((all_sentences, embeddings), f)



Using endpoint : https://twelve-courses.openai.azure.com
Using model    : text-embedding-3-large-context-course-2026-afthibara
Embedding 3101 sentences …
Embedding matrix shape: (3101, 3072)


In [13]:
## Cell 4 — Find the 5 sentences most similar to the query

QUERY = "Haaland scored once again."
QUERY = "What did Haaland do in the match?"

def cosine_similarity(matrix, vector):
    """Cosine similarity between every row in matrix and a single vector."""
    matrix_norms = np.linalg.norm(matrix, axis=1, keepdims=True)
    vector_norm = np.linalg.norm(vector)
    return (matrix @ vector) / (matrix_norms.flatten() * vector_norm + 1e-10)

# Embed the query
query_embedding = embed_texts([QUERY])[0]

# Rank sentences
similarities = cosine_similarity(embeddings, query_embedding)
top5_indices = np.argsort(similarities)[::-1][:5]

print(f'Top 5 sentences most similar to: "{QUERY}"\n')
for rank, idx in enumerate(top5_indices, 1):
    print(f"{rank}. (score={similarities[idx]:.4f})\n   {all_sentences[idx]}\n")


Top 5 sentences most similar to: "What did Haaland do in the match?"

1. (score=0.6637)
   Haaland was decisive.

2. (score=0.5890)
   He found Haaland, who simultaneously wrestled Gabriel away from the ball, while in the same movement spanking it low into the corner.

3. (score=0.5863)
   It soon petered out. 82 min Haaland’s goal was his 23rd in the Premier League this season.

4. (score=0.5634)
   Haaland scored the winner, and in process confirmed his unicorn status in the Premier League as a pure goalscorer, miles out on his own on the numbers, the capocapocannonieri.

5. (score=0.5507)
   Credit to Haaland for this.

